In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Loading Raw Data").getOrCreate()


Load the Bronze Table

In [0]:
bronze_df = spark.table("workspace.default.bronze_anime")
display(bronze_df)

Select Only Required/Analytical Columns

In [0]:
silver_df = bronze_df.select(
  "mal_id",
  "title",
  "title_english",
  "type",
  "source",
  "episodes",
  "status",
  "duration",
  "rating",
  "score",
  "scored_by",
  "rank",
  "popularity",
  "members",
  "favorites",
  "season",
  "year",
  "studios",
  "genres",
  "themes",
  "demographics"
)

In [0]:
display(silver_df)

In [0]:
silver_df = silver_df.filter((silver_df.year>=2010) & (silver_df.year<=2026))

In [0]:
display(silver_df)

In [0]:
display(silver_df.describe().toPandas())

In [0]:
display(silver_df.describe().toPandas().set_index('summary').T)

In [0]:
from pyspark.sql.functions import col, sum, when, round

total = silver_df.count()

silver_df.select([
    round(
        sum(when(col(c).isNull(), 1).otherwise(0)) / total * 100, 2
    ).alias(c)
    for c in silver_df.columns
]).display()

In [0]:
from pyspark.sql.functions import col, when

# Get median for episodes only
median_episodes = int(silver_df.approxQuantile("episodes", [0.5], 0.01)[0])

silver_df = silver_df \
    .withColumn("title_english",
        when(col("title_english").isNull(), col("title"))
        .otherwise(col("title_english"))) \
    .withColumn("score",
        when(col("score").isNull(), 0)
        .otherwise(col("score"))) \
    .withColumn("scored_by",
        when(col("scored_by").isNull(), 0)
        .otherwise(col("scored_by"))) \
    .fillna({
        "demographics" : "Unknown",
        "themes"       : "None",
        "studios"      : "Unknown",
        "rating"       : "Unknown",
        "rank"         : 0,
        "genres"       : "Unknown",
        "episodes"     : median_episodes
    })

display(silver_df)

In [0]:
from pyspark.sql.functions import col, sum, when, round

total = silver_df.count()

silver_df.select([
    round(
        sum(when(col(c).isNull(), 1).otherwise(0)) / total * 100, 2
    ).alias(c)
    for c in silver_df.columns
]).display()

In [0]:
from pyspark.sql.functions import col

silver_df = silver_df \
    .withColumn("mal_id",     col("mal_id").cast("integer")) \
    .withColumn("episodes",   col("episodes").cast("integer")) \
    .withColumn("score",      col("score").cast("double")) \
    .withColumn("scored_by",  col("scored_by").cast("integer")) \
    .withColumn("rank",       col("rank").cast("integer")) \
    .withColumn("popularity", col("popularity").cast("integer")) \
    .withColumn("members",    col("members").cast("integer")) \
    .withColumn("favorites",  col("favorites").cast("integer")) \
    .withColumn("year",       col("year").cast("integer"))

In [0]:
from pyspark.sql.functions import trim, initcap

silver_df = silver_df \
    .withColumn("title",        trim(col("title"))) \
    .withColumn("title_english",trim(col("title_english"))) \
    .withColumn("type",         trim(col("type"))) \
    .withColumn("source",       trim(col("source"))) \
    .withColumn("status",       trim(col("status"))) \
    .withColumn("season",       trim(col("season")))

In [0]:
silver_df = silver_df.dropDuplicates(["mal_id"])

In [0]:
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_anime")
